# Extended W_rib Sweep — Top 3 Results

Plots band diagrams and ng(λ) curves for the top 3 configurations from `sweep_wrib_extended.py`.

**Structure**: Y-symmetric fishbone grating, 71nm partial etch, SiO2 cladding, polysilicon (n=3.48)  
**Target**: Flat group index ng ≈ 6–7 on band 6 (y-even sector) near 1550nm

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import AutoMinorLocator

# --- Load data ---
NPZ_PATH  = 'output/sweep_wrib_extended.npz'
JSON_PATH = 'output/sweep_wrib_extended.json'

data = np.load(NPZ_PATH, allow_pickle=False)
with open(JSON_PATH) as f:
    summary = json.load(f)

# Constants
N_SIO2    = 1.44
WL_TARGET = 1550.0
TARGET_BAND = 6
NG_LO, NG_HI = 6.0, 7.0   # target window
PLATEAU_NG_LO = 5.0        # shade the flat plateau region

N_TOP = 3
COLORS = ['#e41a1c', '#377eb8', '#4daf4a']

print(f'Loaded {len(summary)} ranked configurations')
print(f'Plotting top {N_TOP}')

In [ ]:
def compute_ng(f_band, k_x):
    df_dk = np.gradient(f_band, k_x)
    df_dk = np.where(np.abs(df_dk) < 1e-12, np.nan, df_dk)
    return 1.0 / df_dk

# --- Summary table ---
print(f'{"#":>3}  {"h_spine":>8}  {"h_rib":>7}  {"W_rib":>7}  '
      f'{"a_ref":>7}  {"a_align":>8}  {"gap(nm)":>8}  '
      f'{"ng":>6}  {"σ(ng)":>7}  {"BW(nm)":>7}  {"λc(nm)":>8}')
print('-' * 95)
for s in summary[:10]:
    print(f'{s["h_spine"]:>12.3f}  {s["h_rib"]:>7.3f}  {s["W_rib"]:>7.3f}  '
          f'{s["a_nm_ref"]:>7.1f}  {s["a_nm_aligned"]:>8.1f}  {s["gap_nm"]:>8.1f}  '
          f'{s["ng_plateau"]:>6.3f}  {s["ng_std"]:>7.4f}  {s["bw_nm"]:>7.1f}  {s["wl_center"]:>8.1f}')

## Band Diagrams — Top 3

In [ ]:
fig, axes = plt.subplots(1, N_TOP, figsize=(5.5 * N_TOP, 5.5), sharey=True)

F_LO, F_HI = 0.20, 0.42

for i, ax in enumerate(axes):
    s = summary[i]
    all_freqs = data[f'top{i}_all_freqs']
    k_x       = data[f'top{i}_k_x']
    a_nm      = float(data[f'top{i}_a_nm'])
    nb        = all_freqs.shape[1]
    color     = COLORS[i]

    f_1550 = a_nm / WL_TARGET

    # Light cone (SiO2)
    k_lc = np.linspace(k_x[0], k_x[-1], 200)
    f_lc = k_lc / N_SIO2
    ax.fill_between(k_lc, F_LO, np.minimum(f_lc, F_HI),
                    alpha=0.10, color='gray', label='Light cone')
    ax.plot(k_lc, f_lc, color='gray', lw=0.7, ls='--')

    # All bands (gray)
    for b in range(nb):
        if b != TARGET_BAND:
            ax.plot(k_x, all_freqs[:, b], color='lightgray', lw=0.9, zorder=1)

    # Target band 6 (colored)
    fb = all_freqs[:, TARGET_BAND]
    ax.plot(k_x, fb, color=color, lw=2.2, zorder=3, label=f'Band {TARGET_BAND}')

    # Highlight flat plateau region on band 6
    ng = compute_ng(fb, k_x)
    wl = np.where(fb > 0, a_nm / fb, np.nan)
    plateau_mask = (np.isfinite(ng) & (ng >= PLATEAU_NG_LO) & (ng <= NG_HI)
                    & (wl > 1400) & (wl < 1700))
    if np.any(plateau_mask):
        f_pl = fb[plateau_mask]
        ax.axhspan(f_pl.min(), f_pl.max(), color='lime', alpha=0.25, zorder=2,
                   label=f'ng ∈ [{PLATEAU_NG_LO},{NG_HI}]')

    # 1550nm line
    ax.axhline(f_1550, color='green', ls=':', lw=1.2, label='1550 nm', zorder=4)

    ax.set_xlim(k_x[0], k_x[-1])
    ax.set_ylim(F_LO, F_HI)
    ax.set_xlabel('k  (2π/a)', fontsize=11)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.grid(True, alpha=0.25)

    gap_nm = s['gap_nm']
    ax.set_title(
        f'#{i+1}  h_spine={s["h_spine"]:.3f}  h_rib={s["h_rib"]:.3f}  W_rib={s["W_rib"]:.3f}\n'
        f'a={a_nm:.1f} nm  gap={gap_nm:.0f} nm  '
        f'ng={s["ng_plateau"]:.3f}±{s["ng_std"]:.3f}  BW={s["bw_nm"]:.1f} nm',
        fontsize=9
    )
    ax.legend(fontsize=7, loc='upper left')

axes[0].set_ylabel('Frequency  (a/λ)', fontsize=11)
fig.suptitle('Band Diagrams — Top 3 W_rib Sweep  (y-even, 71 nm PE, band 6 highlighted)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('output/wrib_top3_bands.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: output/wrib_top3_bands.png')

## ng(λ) Curves — Top 3 on one panel

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

for i in range(N_TOP):
    s = summary[i]
    all_freqs = data[f'top{i}_all_freqs']
    k_x       = data[f'top{i}_k_x']
    a_nm      = float(data[f'top{i}_a_nm'])
    color     = COLORS[i]

    fb = all_freqs[:, TARGET_BAND]
    ng = compute_ng(fb, k_x)
    wl = np.where(fb > 0, a_nm / fb, np.nan)

    mask = np.isfinite(ng) & (ng > 1) & (ng < 25) & (wl > 1350) & (wl < 1750)

    label = (f'#{i+1}  hs={s["h_spine"]:.3f}  hr={s["h_rib"]:.3f}  '
             f'Wr={s["W_rib"]:.3f}  a={a_nm:.0f} nm\n'
             f'       ng={s["ng_plateau"]:.3f}±{s["ng_std"]:.3f}  '
             f'BW={s["bw_nm"]:.1f} nm  gap={s["gap_nm"]:.0f} nm')

    ax.plot(wl[mask], ng[mask], '-o', color=color, lw=2.0, ms=5,
            label=label, alpha=0.9)

    # Mark the flat plateau points
    plateau_mask = mask & (ng >= PLATEAU_NG_LO) & (ng <= NG_HI)
    if np.any(plateau_mask):
        ax.plot(wl[plateau_mask], ng[plateau_mask], 'o', color=color,
                ms=9, mec='k', mew=1.2, zorder=5)

# Target window
ax.axhspan(NG_LO, NG_HI, color='lime', alpha=0.15, label=f'Target ng ∈ [{NG_LO}, {NG_HI}]')
ax.axhline(NG_LO, color='green', ls='--', lw=0.9)
ax.axhline(NG_HI, color='green', ls='--', lw=0.9)
ax.axvline(WL_TARGET, color='gray', ls=':', lw=1.2, label='1550 nm')

ax.set_xlabel('Wavelength (nm)', fontsize=12)
ax.set_ylabel('Group Index  $n_g$', fontsize=12)
ax.set_title('Group Index vs. Wavelength — Top 3 W_rib Sweep  (band 6, y-even)',
             fontsize=12)
ax.set_xlim(1400, 1700)
ax.set_ylim(1, 20)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig('output/wrib_top3_ng_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: output/wrib_top3_ng_curves.png')

## Combined: Band diagram + ng(λ) for each top-3 config

In [ ]:
fig = plt.figure(figsize=(6 * N_TOP, 9))
gs  = gridspec.GridSpec(2, N_TOP, height_ratios=[1.6, 1], hspace=0.35, wspace=0.25)

F_LO, F_HI = 0.20, 0.42

for i in range(N_TOP):
    s = summary[i]
    all_freqs = data[f'top{i}_all_freqs']
    k_x       = data[f'top{i}_k_x']
    a_nm      = float(data[f'top{i}_a_nm'])
    nb        = all_freqs.shape[1]
    color     = COLORS[i]
    fb        = all_freqs[:, TARGET_BAND]
    ng        = compute_ng(fb, k_x)
    wl        = np.where(fb > 0, a_nm / fb, np.nan)
    f_1550    = a_nm / WL_TARGET

    # --- Top: band diagram ---
    ax_b = fig.add_subplot(gs[0, i])

    k_lc = np.linspace(k_x[0], k_x[-1], 200)
    f_lc = k_lc / N_SIO2
    ax_b.fill_between(k_lc, F_LO, np.minimum(f_lc, F_HI),
                      alpha=0.10, color='gray')
    ax_b.plot(k_lc, f_lc, color='gray', lw=0.7, ls='--')

    for b in range(nb):
        if b != TARGET_BAND:
            ax_b.plot(k_x, all_freqs[:, b], color='lightgray', lw=0.9)
    ax_b.plot(k_x, fb, color=color, lw=2.2, label=f'Band {TARGET_BAND}')

    plateau_mask = (np.isfinite(ng) & (ng >= PLATEAU_NG_LO) & (ng <= NG_HI)
                    & (wl > 1400) & (wl < 1700))
    if np.any(plateau_mask):
        f_pl = fb[plateau_mask]
        ax_b.axhspan(f_pl.min(), f_pl.max(), color='lime', alpha=0.25,
                     label=f'ng ∈ [{PLATEAU_NG_LO},{NG_HI}]')

    ax_b.axhline(f_1550, color='green', ls=':', lw=1.2, label='1550 nm')
    ax_b.set_xlim(k_x[0], k_x[-1])
    ax_b.set_ylim(F_LO, F_HI)
    ax_b.set_xlabel('k  (2π/a)', fontsize=10)
    if i == 0:
        ax_b.set_ylabel('Frequency  (a/λ)', fontsize=10)
    ax_b.xaxis.set_minor_locator(AutoMinorLocator())
    ax_b.yaxis.set_minor_locator(AutoMinorLocator())
    ax_b.grid(True, alpha=0.25)
    ax_b.legend(fontsize=7, loc='upper left')
    ax_b.set_title(
        f'#{i+1}  hs={s["h_spine"]:.3f}  hr={s["h_rib"]:.3f}  Wr={s["W_rib"]:.3f}\n'
        f'a={a_nm:.1f} nm  gap={s["gap_nm"]:.0f} nm\n'
        f'ng={s["ng_plateau"]:.3f}±{s["ng_std"]:.3f}  BW={s["bw_nm"]:.1f} nm',
        fontsize=9
    )

    # --- Bottom: ng(λ) ---
    ax_n = fig.add_subplot(gs[1, i])

    mask = np.isfinite(ng) & (ng > 1) & (ng < 25) & (wl > 1350) & (wl < 1750)
    ax_n.plot(wl[mask], ng[mask], '-o', color=color, lw=1.8, ms=4)

    # Bold dots on flat plateau
    pl_mask = mask & (ng >= PLATEAU_NG_LO) & (ng <= NG_HI)
    if np.any(pl_mask):
        ax_n.plot(wl[pl_mask], ng[pl_mask], 'o', color=color,
                  ms=8, mec='k', mew=1.2, zorder=5)

    ax_n.axhspan(NG_LO, NG_HI, color='lime', alpha=0.15)
    ax_n.axhline(NG_LO, color='green', ls='--', lw=0.8)
    ax_n.axhline(NG_HI, color='green', ls='--', lw=0.8)
    ax_n.axvline(WL_TARGET, color='gray', ls=':', lw=1.0)
    ax_n.set_xlabel('Wavelength (nm)', fontsize=10)
    if i == 0:
        ax_n.set_ylabel('Group Index  $n_g$', fontsize=10)
    ax_n.set_xlim(1400, 1700)
    ax_n.set_ylim(1, 20)
    ax_n.xaxis.set_minor_locator(AutoMinorLocator())
    ax_n.yaxis.set_minor_locator(AutoMinorLocator())
    ax_n.grid(True, alpha=0.3)

fig.suptitle('Top 3 W_rib Sweep: Band Diagram + ng(λ)  [y-even, 71 nm PE, band 6]',
             fontsize=12, y=1.01)
plt.savefig('output/wrib_top3_combined.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: output/wrib_top3_combined.png')